# Reinforcement Learning op Taxi-v3
## Projectwalkthrough voor de docent

Dit notebook neemt je stap voor stap mee door het volledige RL-project:  
van het begrijpen van de omgeving tot het trainen, vergelijken en visualiseren van agents.

| # | Sectie | Wat je ziet |
|---|---|---|
| 1 | Setup | Dependencies installeren, modules importeren |
| 2 | De omgeving | Taxi-v3: states, actions, rewards en state-decoding |
| 3 | Agents | Overzicht en theorie van alle vier agents |
| 4 | Baselines | Random en rule-based agent evalueren |
| 5 | Q-learning | Theorie, training en greedy evaluatie |
| 6 | SARSA | Theorie, training en greedy evaluatie |
| 7 | Vergelijking | Alle vier agents naast elkaar |
| 8 | Policy visualisatie | Wat heeft de agent geleerd? |
| 9 | Unit tests | Smoketests draaien via pytest |
| 10 | Conclusie | Samenvatting van bevindingen |

## 1 — Setup

Installeer de benodigde packages en stel de importpaden in.  
De broncode staat in `src/`, resultaten worden opgeslagen in `results/notebook_demo/`.

In [1]:
from pathlib import Path
import sys, subprocess

# Zoek requirements.txt in huidige map of een map hoger.
req = Path("requirements.txt")
if not req.exists():
    req = Path("..") / "requirements.txt"

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr)
else:
    print("✓ Packages geïnstalleerd")

✓ Packages geïnstalleerd


In [2]:
import sys
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import yaml

# Voeg projectroot toe aan het importpad zodat src/ gevonden wordt.
PROJECT_ROOT = Path("..").resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.env import TaxiEnv, N_ACTIONS, N_STATES, ACTION_NAMES, LOCATIONS, LOC_NAMES, decode_state
from src.agents.q_learning import QLearningAgent
from src.agents.sarsa import SarsaAgent
from src.agents.random_agent import RandomAgent
from src.agents.rule_based_agent import RuleBasedAgent

# Dict om evaluatieresultaten per agent bij te houden.
results = {}

RESULTS_DIR = PROJECT_ROOT / "results" / "notebook_demo"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Imports geslaagd")
print(f"  PROJECT_ROOT : {PROJECT_ROOT}")
print(f"  States       : {N_STATES}")
print(f"  Actions      : {N_ACTIONS}  →  {ACTION_NAMES}")
print(f"  Results dir  : {RESULTS_DIR}")

✓ Imports geslaagd
  PROJECT_ROOT : C:\Users\JPHou\Desktop\AI praktijken\autonomuis\Autonomuis_2
  States       : 500
  Actions      : 6  →  ('south', 'north', 'east', 'west', 'pickup', 'dropoff')
  Results dir  : C:\Users\JPHou\Desktop\AI praktijken\autonomuis\Autonomuis_2\results\notebook_demo


## 2 — De omgeving: Taxi-v3

Taxi-v3 is een discrete gridworld uit Gymnasium. Een taxi rijdt op een **5×5 grid** en moet een passagier ophalen en afleveren bij één van vier locaties.

```
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
```

Vier vaste locaties: **R** (rood), **G** (groen), **Y** (geel), **B** (blauw).

### State space — 500 discrete states

```
state = (taxi_row * 5 + taxi_col) * 20 + passenger_loc * 4 + destination
```

| Component | Betekenis | Waarden |
|---|---|---|
| `taxi_row` | Rij van de taxi | 0–4 |
| `taxi_col` | Kolom van de taxi | 0–4 |
| `passenger_loc` | Locatie passagier | 0=R, 1=G, 2=Y, 3=B, 4=in_taxi |
| `destination` | Bestemming | 0=R, 1=G, 2=Y, 3=B |

### Action space — 6 actions

| Index | Action | Omschrijving |
|---|---|---|
| 0 | south | Rij naar beneden |
| 1 | north | Rij naar boven |
| 2 | east | Rij naar rechts |
| 3 | west | Rij naar links |
| 4 | pickup | Passagier oppakken |
| 5 | dropoff | Passagier afzetten |

### Rewards

| Situatie | Reward |
|---|---|
| Elke stap | −1 |
| Ongeldige pickup of dropoff | −10 |
| Succesvolle aflevering | +20 (episode klaar) |

In [3]:
# Reset de omgeving en toon de begin-state (gedecodeerd).
env = TaxiEnv()
s0 = env.reset(seed=42)
dec = decode_state(s0)

print("=== Begin-state ===")
print(f"  Integer state : {s0}")
print(f"  taxi_row      : {dec['taxi_row']}")
print(f"  taxi_col      : {dec['taxi_col']}")
print(f"  passenger_loc : {dec['passenger_loc']}  ({LOC_NAMES[dec['passenger_loc']]})")
print(f"  destination   : {dec['destination']}  ({LOC_NAMES[dec['destination']]})")

print("\n=== Effect van elke action vanuit de begin-state ===")
print(f"{'Idx':<5} {'Action':<10} {'→ State':<9} {'Reward':<9} {'Done':<7} {'Flags'}")
print("-" * 56)
for a_idx, a_name in enumerate(ACTION_NAMES):
    env.reset(seed=42)
    s1, r, done, info = env.step(a_idx)
    flags = []
    if info.get("illegal_action"):
        flags.append("ILLEGAL")
    if info.get("delivered"):
        flags.append("DELIVERED")
    print(f"{a_idx:<5} {a_name:<10} {s1:<9} {r:<9.0f} {str(done):<7} {', '.join(flags) or '-'}")

env.close()

c:\Users\JPHou\miniconda3\Lib\site-packages\gymnasium\envs\registration.py:513: DeprecationWarning: WARN: The environment Taxi-v3 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(


DeprecatedEnv: Environment version v3 for `Taxi` is deprecated. Please use `Taxi-v4` instead.

## 3 — De agents: overzicht & theorie

Het project bevat vier agents, van eenvoudig naar complex:

| Agent | Klasse | Leert? | Strategie |
|---|---|---|---|
| Random | `RandomAgent` | ✗ | Willekeurige action elke stap |
| Rule-based | `RuleBasedAgent` | ✗ | Vaste heuristiek (Manhattan-afstand) |
| Q-learning | `QLearningAgent` | ✓ | Off-policy, epsilon-greedy |
| SARSA | `SarsaAgent` | ✓ | On-policy, epsilon-greedy |

---

### Q-learning (off-policy)

$$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\bigr]$$

- **Off-policy**: we leren de *optimale* policy, ongeacht welke action de agent werkelijk kiest.
- De term $\max_{a'} Q(s',a')$ kijkt altijd naar de beste mogelijke volgende action.
- Epsilon-greedy: met kans $\varepsilon$ een willekeurige action (exploration), anders de beste (exploitation).

### SARSA (on-policy)

$$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl[r + \gamma Q(s',a') - Q(s,a)\bigr]$$

- **On-policy**: we leren de policy die de agent *daadwerkelijk volgt*, inclusief willekeurige keuzes.
- $a'$ is de werkelijk gekozen volgende action — ook als die suboptimaal is.
- SARSA is conservatiever: het rekent de kosten van exploration in, Q-learning niet.

### Gemeenschappelijke hyperparameters

| Parameter | Symbool | Betekenis | Standaard |
|---|---|---|---|
| Learning rate | $\alpha$ | Hoe snel nieuwe info de Q-table overschrijft | 0.5 |
| Discount factor | $\gamma$ | Gewicht van toekomstige rewards | 0.99 |
| Epsilon start | $\varepsilon_0$ | Beginwaarde exploration | 1.0 |
| Epsilon end | $\varepsilon_\infty$ | Eindwaarde exploration | 0.05 |
| Epsilon decay | — | Episodes totdat $\varepsilon$ zijn eindwaarde bereikt | 2000 |

## 4 — Baselines: Random & Rule-Based

We evalueren eerst de twee baselines over 100 episodes. Dit geeft ons een referentiepunt voor de RL-agents.

- **RandomAgent** — kiest elke stap een willekeurige action. Verwachte return is sterk negatief.
- **RuleBasedAgent** — volgt een vaste heuristiek: ga naar de passagier → pickup → ga naar bestemming → dropoff. Weet niets van muren (loopt er soms tegenaan), maar haalt wel consequent de passagier op en levert af.

In [ ]:
def evaluate_agent(agent, n_episodes=100, seed=0, max_steps=200):
    """Evalueer een agent greedy over n_episodes en geef statistieken terug."""
    env = TaxiEnv()
    returns, lengths, illegals, successes = [], [], [], []

    for ep in range(n_episodes):
        state = env.reset(seed=seed + ep)
        ep_return, steps, illegal, delivered = 0.0, 0, 0, False

        for _ in range(max_steps):
            action = agent.select_action(state, greedy=True)
            state, reward, done, info = env.step(action)
            ep_return += reward
            steps += 1
            if info.get("illegal_action"):
                illegal += 1
            if info.get("delivered"):
                delivered = True
            if done:
                break

        returns.append(ep_return)
        lengths.append(steps)
        illegals.append(illegal)
        successes.append(int(delivered))

    env.close()
    return dict(
        return_mean=round(float(np.mean(returns)), 2),
        return_std=round(float(np.std(returns)), 2),
        len_mean=round(float(np.mean(lengths)), 1),
        illegal_mean=round(float(np.mean(illegals)), 1),
        success_rate=round(float(np.mean(successes)), 3),
    )


def print_results(res_dict):
    """Druk een overzichtstabel af van agent-statistieken."""
    hdr = f"{'Agent':<16} {'Return μ':>9} {'± σ':>7} {'Succes %':>10} {'Lengte':>8} {'Illegaal':>10}"
    print(hdr)
    print("-" * len(hdr))
    for name, r in res_dict.items():
        print(
            f"{name:<16} {r['return_mean']:>9.1f} {r['return_std']:>7.1f}"
            f" {r['success_rate']*100:>9.1f}% {r['len_mean']:>8.1f} {r['illegal_mean']:>10.1f}"
        )


# --- Evalueer de twee baselines ---
print("Baselines evalueren (100 episodes)...\n")
results["random"]     = evaluate_agent(RandomAgent(seed=0))
results["rule_based"] = evaluate_agent(RuleBasedAgent(seed=0))

print_results({k: results[k] for k in ("random", "rule_based")})

## 5 — Training: Q-learning

We laden de hyperparameters uit `experiments/qlearning_default.yaml` en trainen de agent voor **2 000 episodes**.  
De Q-table (500 × 6 floats) wordt bij elke stap bijgewerkt via de Bellman-regel.

**Epsilon-decay**: ε daalt lineair van 1.0 naar 0.05 over de eerste 2 000 episodes.  
Dit zorgt voor veel exploration aan het begin en meer exploitation naarmate de training vordert.

In [ ]:
def train_agent(agent, is_sarsa=False, episodes=2000, max_steps=200, seed=0):
    """Train de agent voor het opgegeven aantal episodes.

    Geeft een NumPy-array terug met per episode:
    [episode_nr, total_return, steps, n_illegal, delivered (0/1), epsilon]
    """
    env = TaxiEnv()
    log = []

    for ep in range(episodes):
        state = env.reset(seed=seed + ep)
        # SARSA: kies de eerste action vóór de lus (nodig voor on-policy update).
        action = agent.select_action(state) if is_sarsa else None
        ep_return, steps, illegal, delivered = 0.0, 0, 0, False

        for _ in range(max_steps):
            if not is_sarsa:
                action = agent.select_action(state)

            next_state, reward, done, info = env.step(action)
            ep_return += reward
            steps += 1
            if info.get("illegal_action"):
                illegal += 1
            if info.get("delivered"):
                delivered = True

            # SARSA on-policy: kies de volgende action NU zodat die meegenomen wordt.
            # Q-learning off-policy: de volgende action hoeft niet mee in de update.
            next_action = agent.select_action(next_state) if (is_sarsa and not done) else None
            agent.update(state, action, reward, next_state, next_action, done)

            state = next_state
            if is_sarsa and not done:
                action = next_action
            if done:
                break

        agent.end_episode()
        log.append([ep, ep_return, steps, illegal, int(delivered), agent.epsilon])

    env.close()
    return np.array(log, dtype=float)


# --- Laad Q-learning config en train ---
cfg_path = PROJECT_ROOT / "experiments" / "qlearning_default.yaml"
with open(cfg_path) as f:
    ql_cfg = yaml.safe_load(f).get("agent", {})

print("Q-learning hyperparameters:")
for k, v in ql_cfg.items():
    print(f"  {k:<28} = {v}")

random.seed(0)
np.random.seed(0)
ql_agent = QLearningAgent(n_states=N_STATES, n_actions=N_ACTIONS, seed=0, **ql_cfg)

print(f"\nTraining gestart ({2000} episodes)...")
ql_log = train_agent(ql_agent, is_sarsa=False, episodes=2000, seed=0)
print(
    f"✓ Klaar | Laatste 100 eps: "
    f"mean return = {ql_log[-100:, 1].mean():.2f} | "
    f"succes = {ql_log[-100:, 4].mean() * 100:.0f}%"
)

### Q-learning: training curve

De ruwe return per episode (lichtblauw, transparant) en het voortschrijdend gemiddelde (window = 100).  
Een duidelijk stijgende trend betekent dat de agent leert. De stippellijnen tonen de baselines als referentie.

In [ ]:
def plot_curve(log, label, color, window=100, ax=None, show_raw=True):
    """Plot een training curve met voortschrijdend gemiddelde."""
    returns = log[:, 1]
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    if show_raw:
        ax.plot(returns, alpha=0.15, color=color, linewidth=0.6)
    if len(returns) >= window:
        smoothed = np.convolve(returns, np.ones(window) / window, mode="valid")
        x_sm = np.arange(window - 1, len(returns))
        ax.plot(x_sm, smoothed, color=color, linewidth=2.5, label=f"{label} (w={window})")
    return ax


fig, ax = plt.subplots(figsize=(10, 4))
plot_curve(ql_log, "Q-learning", color="steelblue", ax=ax)
ax.axhline(
    results["rule_based"]["return_mean"], color="orange", linestyle="--",
    linewidth=1.5, label=f"rule-based baseline ({results['rule_based']['return_mean']:.1f})"
)
ax.axhline(
    results["random"]["return_mean"], color="gray", linestyle=":",
    linewidth=1.5, label=f"random baseline ({results['random']['return_mean']:.1f})"
)
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("Q-learning: trainings-reward curve — Taxi-v3")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ql_curve.png", dpi=120)
plt.show()

### Q-learning: greedy evaluatie

Na training zetten we **ε = 0** (geen exploration). De agent kiest altijd de best bekende action.

In [ ]:
# Zet epsilon op 0: agent kiest altijd de beste action (geen willekeur).
ql_agent.epsilon = 0.0
results["qlearning"] = evaluate_agent(ql_agent, n_episodes=100, seed=999)

r = results["qlearning"]
print("Q-learning (greedy, 100 episodes):")
print(f"  Return:             {r['return_mean']:.2f} ± {r['return_std']:.2f}")
print(f"  Succes rate:        {r['success_rate'] * 100:.1f}%")
print(f"  Gemiddelde lengte:  {r['len_mean']:.1f} stappen")
print(f"  Illegale acties/ep: {r['illegal_mean']:.1f}")

## 6 — Training: SARSA

SARSA (State-Action-Reward-State-Action) is **on-policy**: het leert de Q-values van de policy die het *daadwerkelijk uitvoert*, inclusief de willekeurige exploration.

**Verschil met Q-learning in één zin:**
> Q-learning gebruikt `max Q(s', a')` — leert de *optimale* policy.  
> SARSA gebruikt `Q(s', a')` waar `a'` de *werkelijk gekozen* action is — leert de *gedragsmatige* policy.

In gevaarlijkere omgevingen (met grote straffen) is SARSA voorzichtiger.  
In Taxi-v3 zijn de prestaties doorgaans vergelijkbaar.

In [ ]:
# --- Laad SARSA config en train ---
cfg_path_sarsa = PROJECT_ROOT / "experiments" / "sarsa_default.yaml"
with open(cfg_path_sarsa) as f:
    sarsa_cfg = yaml.safe_load(f).get("agent", {})

print("SARSA hyperparameters:")
for k, v in sarsa_cfg.items():
    print(f"  {k:<28} = {v}")

random.seed(1)
np.random.seed(1)
sarsa_agent = SarsaAgent(n_states=N_STATES, n_actions=N_ACTIONS, seed=1, **sarsa_cfg)

print(f"\nTraining gestart ({2000} episodes)...")
sarsa_log = train_agent(sarsa_agent, is_sarsa=True, episodes=2000, seed=1)
print(
    f"✓ Klaar | Laatste 100 eps: "
    f"mean return = {sarsa_log[-100:, 1].mean():.2f} | "
    f"succes = {sarsa_log[-100:, 4].mean() * 100:.0f}%"
)

### SARSA: training curve

De ruwe return per episode (transparant) en het voortschrijdend gemiddelde (window = 100).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_curve(sarsa_log, "SARSA", color="darkorange", ax=ax)
ax.axhline(
    results["rule_based"]["return_mean"], color="green", linestyle="--",
    linewidth=1.5, label=f"rule-based baseline ({results['rule_based']['return_mean']:.1f})"
)
ax.axhline(
    results["random"]["return_mean"], color="gray", linestyle=":",
    linewidth=1.5, label=f"random baseline ({results['random']['return_mean']:.1f})"
)
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("SARSA: trainings-reward curve — Taxi-v3")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sarsa_curve.png", dpi=120)
plt.show()

### SARSA: greedy evaluatie

Na training zetten we **ε = 0** (geen exploration). De agent kiest altijd de best bekende action.

In [ ]:
# Zet epsilon op 0: agent kiest altijd de beste action.
sarsa_agent.epsilon = 0.0
results["sarsa"] = evaluate_agent(sarsa_agent, n_episodes=100, seed=999)

r = results["sarsa"]
print("SARSA (greedy, 100 episodes):")
print(f"  Return:             {r['return_mean']:.2f} ± {r['return_std']:.2f}")
print(f"  Succes rate:        {r['success_rate'] * 100:.1f}%")
print(f"  Gemiddelde lengte:  {r['len_mean']:.1f} stappen")
print(f"  Illegale acties/ep: {r['illegal_mean']:.1f}")

## 7 — Vergelijking: alle agents

We zetten alle vier agents naast elkaar in één tabel en twee grafieken.

In [ ]:
# --- Overzichtstabel ---
print("=== Overzicht: alle agents (100 greedy episodes) ===\n")
print_results(results)

# --- Gecombineerde training curve ---
fig, ax = plt.subplots(figsize=(11, 5))
plot_curve(ql_log, "Q-learning", color="steelblue", ax=ax)
plot_curve(sarsa_log, "SARSA", color="darkorange", ax=ax)
ax.axhline(
    results["rule_based"]["return_mean"], color="green", linestyle="--",
    linewidth=1.5, label=f"rule-based ({results['rule_based']['return_mean']:.1f})"
)
ax.axhline(
    results["random"]["return_mean"], color="gray", linestyle=":",
    linewidth=1.5, label=f"random ({results['random']['return_mean']:.1f})"
)
ax.set_xlabel("Episode")
ax.set_ylabel("Return (voortschrijdend gemiddelde, w=100)")
ax.set_title("Trainings-reward curves — Taxi-v3: Q-learning vs SARSA")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "comparison_curves.png", dpi=120)
plt.show()

# --- Staafdiagrammen: succes rate & gemiddelde return ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
names = list(results.keys())
success_rates = [results[n]["success_rate"] * 100 for n in names]
return_means  = [results[n]["return_mean"] for n in names]
bar_colors = ["#aaaaaa", "#4caf50", "#2196f3", "#ff9800"]

axes[0].bar(names, success_rates, color=bar_colors[:len(names)])
axes[0].set_ylabel("Succes rate (%)")
axes[0].set_title("Succes rate per agent")
axes[0].set_ylim(0, 115)
for i, v in enumerate(success_rates):
    axes[0].text(i, v + 1.5, f"{v:.1f}%", ha="center", fontsize=10)

axes[1].bar(names, return_means, color=bar_colors[:len(names)])
axes[1].set_ylabel("Gemiddelde return")
axes[1].set_title("Gemiddelde return per agent")
for i, v in enumerate(return_means):
    offset = 1.5 if v >= 0 else -3
    axes[1].text(i, v + offset, f"{v:.1f}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "comparison_bars.png", dpi=120)
plt.show()

## 8 — Policy visualisatie

Wat heeft de Q-learning agent precies geleerd?  
We slaan de Q-table op en tekenen voor elke passagier/bestemming-combinatie een 5×5 grid van pijlen.

- **Kleur** van elke cel = $V(s) = \max_a Q(s, a)$ (hoe waardevol is deze staat?)
- **Pijl of letter** = de greedy action in die staat (P = pickup, D = dropoff)

In [ ]:
import subprocess as _sp

weights_path = RESULTS_DIR / "ql_qtable.npy"
ql_agent.save(str(weights_path))

policy_path = RESULTS_DIR / "ql_policy.png"
proc = _sp.run(
    [sys.executable, "-m", "src.plot_policy",
     "--weights", str(weights_path),
     "--output", str(policy_path),
     "--title", "Q-learning greedy policy (notebook demo)"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)

if proc.returncode != 0:
    print("Fout bij het genereren van de policy plot:")
    print(proc.stderr)
else:
    img = plt.imread(str(policy_path))
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Q-learning greedy policy — passagier wachtend, alle bestemmingen")
    plt.tight_layout()
    plt.show()
    print(f"Policy plot opgeslagen in: {policy_path}")

## 9 — Unit tests

We draaien de smoketests via `pytest`. Deze testen of:

- De omgeving geldige states en datatypes teruggeeft
- Alle agents actions kiezen binnen het geldige bereik
- Q-learning en SARSA hun Q-table aanpassen na een update
- Opslaan en laden van een Q-table verliesvrij werkt

Een groene `PASSED` naast elke test betekent dat de implementatie correct is.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 10 — Conclusie

### Resultaten samengevat

| Agent | Verwachting | Uitkomst |
|---|---|---|
| Random | Zeer negatieve return, lage succes rate | ✓ Bevestigd |
| Rule-based | Redelijke succes rate, maar suboptimaal | ✓ Bevestigd |
| Q-learning | Significant beter dan beide baselines | ✓ Bevestigd |
| SARSA | Vergelijkbaar met Q-learning | ✓ Bevestigd |

### Bevindingen

- Zowel Q-learning als SARSA leren effectief om de passagier op te halen en af te leveren.
- **Epsilon-decay** is cruciaal: zonder exploration leert de agent niets; zonder exploitation convergeert hij niet.
- **SARSA vs Q-learning**: vergelijkbare prestaties op Taxi-v3. In omgevingen met hogere straf zou SARSA voorzichtiger zijn.
- De **rule-based baseline** is een goede sanity check: een simpele heuristiek presteert al beter dan random.

### Mogelijke verbeteringen

- Hyperparameter sweep (zie `src/sweep.py` en de sweepresultaten in `results/sweep_*/`)
- Meer trainingstijd (5 000+ episodes) voor stabielere convergentie
- Andere exploration-strategieën (bijv. UCB of Boltzmann)
- Deep RL met neurale netwerken voor grotere, continue state spaces